# Train Conv-VAE-Neo MultiView Concat

Train one joint VAE over synchronized camera views concatenated at their full width.

In [ ]:
import pathlib
import pprint
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import torch
from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"
from sensorprocessing.conv_vae_neo import ConvVAENeoLoss
from sensorprocessing.conv_vae_neo_multiview_concat import (
    ConvVAENeoMultiViewConcatModel, make_dataloaders, train,
)

## Exp/run parameters

In [ ]:
creation_style = "exist-ok"
expruns_path = None
results_path = None
epochs = None
experiment = "sensorprocessing_conv_vae_neo_multiview_concat"
run = "sp_vae_neo_multiview_concat_128_256px"

In [ ]:
if expruns_path:
    expruns_path = pathlib.Path(expruns_path)
    if not expruns_path.exists():
        raise FileNotFoundError(expruns_path)
    Config().set_exprun_path(expruns_path)
    Config().copy_experiment(experiment)
    Config().copy_experiment("demonstration")
if results_path:
    results_path = pathlib.Path(results_path)
    if not results_path.exists():
        raise FileNotFoundError(results_path)
    Config().set_results_path(results_path)

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
pprint.pprint(exp)

## Train or resume

In [ ]:
exp.start_timer("training")
try:
    model = train(exp, epochs=epochs)
finally:
    exp.end_timer("training")

## Load the best model and inspect reconstructions

In [ ]:
device = Config().runtime["device"]
checkpoint_path = pathlib.Path(exp["data_dir"], "checkpoints", "best_model.pth")
# To load an intermediate checkpoint instead, replace checkpoint_path with:
# checkpoint_path = pathlib.Path(exp["data_dir"], "checkpoints", "epoch_000100.pth")
if not checkpoint_path.is_file():
    raise FileNotFoundError(checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model = ConvVAENeoMultiViewConcatModel(exp).to(device)
state = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state, strict=True)
model.eval()
print(f"Loaded checkpoint: {checkpoint_path}")

In [ ]:
_, validation_loader = make_dataloaders(exp)
views = next(iter(validation_loader))
views = [view.to(device) for view in views]
with torch.no_grad():
    output = model(views)
composite = model.compose_views(views)
components = ConvVAENeoLoss(exp).components(output, composite)
print({name: float(value) for name, value in components.items()})

original_views = model.split_composite(composite.cpu())
reconstructed_views = model.split_composite(output[0].cpu())
count = min(4, composite.size(0))
fig, axes = plt.subplots(2 * model.num_views, count, figsize=(3 * count, 3 * 2 * model.num_views), squeeze=False)
for view_index in range(model.num_views):
    for sample_index in range(count):
        axes[2 * view_index, sample_index].imshow(original_views[view_index][sample_index].permute(1, 2, 0))
        axes[2 * view_index, sample_index].axis("off")
        axes[2 * view_index + 1, sample_index].imshow(reconstructed_views[view_index][sample_index].permute(1, 2, 0))
        axes[2 * view_index + 1, sample_index].axis("off")
    axes[2 * view_index, 0].set_title(f"{model.cameras[view_index]} original")
    axes[2 * view_index + 1, 0].set_title(f"{model.cameras[view_index]} reconstruction")
fig.tight_layout()
figure_path = pathlib.Path(exp["data_dir"], "training_reconstructions.png")
fig.savefig(figure_path, bbox_inches="tight")
plt.show()